In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

In [2]:
#Load
data_5min2016 = pd.read_csv('resampled_5min_2016_with_features.csv')
data_5min2017 = pd.read_csv('resampled_5min_2017_with_features.csv')
data_5min2018 = pd.read_csv('resampled_5min_2018_with_features.csv')
data_5min2019 = pd.read_csv('resampled_5min_2019_with_features.csv')
data_5min2020 = pd.read_csv('resampled_5min_2020_with_features.csv')
data_5min2021 = pd.read_csv('resampled_5min_2021_with_features.csv')
data_5min2022 = pd.read_csv('resampled_5min_2022_with_features.csv')
data_5min2023 = pd.read_csv('resampled_5min_2023_with_features.csv')


In [3]:
!pip install ta
import ta

for year_data in [data_5min2016, data_5min2017, data_5min2018, data_5min2019, data_5min2020, data_5min2021, data_5min2022]:
    # Simple Moving Averages
    year_data['SMA_5'] = ta.trend.sma_indicator(year_data['Mid_Price_1_last'], window=5)
    year_data['SMA_20'] = ta.trend.sma_indicator(year_data['Mid_Price_1_last'], window=20)

    # Exponential Moving Averages
    year_data['EMA_5'] = ta.trend.ema_indicator(year_data['Mid_Price_1_last'], window=5)
    year_data['EMA_20'] = ta.trend.ema_indicator(year_data['Mid_Price_1_last'], window=20)

    # RSI
    year_data['RSI'] = ta.momentum.rsi(year_data['Mid_Price_1_last'], window=14)

    # MACD
    year_data['MACD'] = ta.trend.macd_diff(year_data['Mid_Price_1_last'], window_slow=26, window_fast=12, window_sign=9)

    # Bollinger Bands
    bollinger = ta.volatility.BollingerBands(close=year_data['Mid_Price_1_last'], window=20, window_dev=2)
    year_data['Bollinger_High'] = bollinger.bollinger_hband()
    year_data['Bollinger_Low'] = bollinger.bollinger_lband()

    # ATR
    year_data['ATR'] = ta.volatility.average_true_range(
        high=year_data['Mid_Price_1_max'], 
        low=year_data['Mid_Price_1_min'], 
        close=year_data['Mid_Price_1_last'], 
        window=14
    )


for year_data in [data_5min2016, data_5min2017, data_5min2018, data_5min2019, data_5min2020, data_5min2021, data_5min2022]:
    year_data.dropna(inplace=True)


In [4]:
# Set up X and y for each year
for year_data in [data_5min2016, data_5min2017, data_5min2018, data_5min2019, data_5min2020, data_5min2021, data_5min2022]:
    year_data['Direction'] = year_data['Mid_Price_1_last'].diff().apply(lambda x: 1 if x > 0 else -1).shift(-1)
    year_data.dropna(inplace=True)  # Remove rows with NaN

    # Select features and drop non-feature columns
    X = year_data.drop(columns=['Mid_Price_1_last', 'Time (sec)', 'Date', 'Direction'])
    y = year_data['Direction']

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

start_time = time.time()

# Grid
param_grid = {
    'n_estimators': [100, 150, 200 ],   
    'criterion': ['gini'],
    'max_depth': [5, 10, 15],  
    'min_samples_split': [2, 5, 10], 
    'min_samples_leaf': [1, 2, 4],    
    'max_features': ['auto', 'sqrt'],  
    'bootstrap': [True, False]         
}


# 2016-2022
data_years = [data_5min2016, data_5min2017, data_5min2018, data_5min2019, data_5min2020, data_5min2021, data_5min2022]

# Loop through rolling window
for i in range(1, len(data_years)):  # Start from year 1 and test on year i+1
    # Train on data from year 1 up to year i
    train_data = pd.concat(data_years[:i])  
    # Test on data from year i
    test_data = data_years[i]
    
    X_train = train_data.drop(columns=['Mid_Price_1_last', 'Time (sec)', 'Date', 'Direction'])
    y_train = train_data['Direction']

    X_test = test_data.drop(columns=['Mid_Price_1_last', 'Time (sec)', 'Date', 'Direction'])
    y_test = test_data['Direction']
    
    X_train.fillna(X_train.mean(), inplace=True)
    X_test.fillna(X_test.mean(), inplace=True)

    # RF classifier
    rf = RandomForestClassifier(random_state=10, n_jobs=-1)
    
    #Timeseriessplit
    tscv = TimeSeriesSplit(n_splits=6)  

    # GridSearchCV 
    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=tscv, n_jobs=-1, verbose=2, scoring='accuracy')
    grid_search.fit(X_train, y_train)

    # Best RF model
    best_rf = grid_search.best_estimator_
    
     # Save best model
    import joblib
    model_filename = f'best_rf_model_{i+1}_to_{i+2}.pkl'  # Save the model for each year-to-year transition
    joblib.dump(best_rf, model_filename)
    
     
    feature_importances = best_rf.feature_importances_
    feature_importance_dict = dict(zip(X_train.columns, feature_importances))
    
    # Feature importances
    print(f"Feature importances for Year {i+1} to {i+2}:")
    for feature, importance in feature_importance_dict.items():
        print(f"{feature}: {importance:.4f}")
    
    # Feature importance
    feature_importances = best_rf.feature_importances_

    # Make predictions on test set with best RF model
    y_pred = best_rf.predict(X_test)

    # Evaluate the model performance
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Display results 
    print(f"Year {i} -> {i+1}")
    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall: {recall * 100:.2f}%")
    print(f"F1 Score: {f1 * 100:.2f}%")
    print('-' * 30)

# Display total time
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Total Time taken: {elapsed_time:.2f} seconds")


Fitting 6 folds for each of 324 candidates, totalling 1944 fits
Feature importances for Year 2 to 3:
Ask Price 1_first: 0.0097
Ask Price 1_max: 0.0078
Ask Price 1_min: 0.0077
Ask Price 1_last: 0.0093
Bid Price 1_first: 0.0094
Bid Price 1_max: 0.0089
Bid Price 1_min: 0.0080
Bid Price 1_last: 0.0078
Ask Price 2_first: 0.0101
Ask Price 2_max: 0.0080
Ask Price 2_min: 0.0077
Ask Price 2_last: 0.0100
Bid Price 2_first: 0.0075
Bid Price 2_max: 0.0072
Bid Price 2_min: 0.0082
Bid Price 2_last: 0.0091
Ask Size 1_sum: 0.0293
Ask Size 1_mean: 0.0310
Bid Size 1_sum: 0.0329
Bid Size 1_mean: 0.0323
Ask Size 2_sum: 0.0291
Ask Size 2_mean: 0.0297
Bid Size 2_sum: 0.0309
Bid Size 2_mean: 0.0310
Price_first: 0.0106
Price_max: 0.0076
Price_min: 0.0084
Price_last: 0.0096
Direction_mean: 0.0376
Mid_Price_1_first: 0.0094
Mid_Price_1_min: 0.0079
Mid_Price_1_max: 0.0087
Mid_Price_2_first: 0.0092
Mid_Price_2_last: 0.0097
Mid_Price_2_min: 0.0093
Mid_Price_2_max: 0.0092
Spread_1_first: 0.0040
Spread_1_last: 0.0021

C:\Users\tuan-\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1245: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Year 5 -> 6
Best Parameters: {'bootstrap': True, 'criterion': 'gini', 'max_depth': 5, 'max_features': 'auto', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 150}
Accuracy: 48.90%
Precision: 23.91%
Recall: 48.90%
F1 Score: 32.11%
------------------------------
Fitting 6 folds for each of 324 candidates, totalling 1944 fits
Feature importances for Year 7 to 8:
Ask Price 1_first: 0.0078
Ask Price 1_max: 0.0053
Ask Price 1_min: 0.0058
Ask Price 1_last: 0.0078
Bid Price 1_first: 0.0076
Bid Price 1_max: 0.0059
Bid Price 1_min: 0.0044
Bid Price 1_last: 0.0088
Ask Price 2_first: 0.0087
Ask Price 2_max: 0.0052
Ask Price 2_min: 0.0030
Ask Price 2_last: 0.0051
Bid Price 2_first: 0.0065
Bid Price 2_max: 0.0064
Bid Price 2_min: 0.0062
Bid Price 2_last: 0.0050
Ask Size 1_sum: 0.0179
Ask Size 1_mean: 0.0231
Bid Size 1_sum: 0.0320
Bid Size 1_mean: 0.0377
Ask Size 2_sum: 0.0282
Ask Size 2_mean: 0.0308
Bid Size 2_sum: 0.0397
Bid Size 2_mean: 0.0439
Price_first: 0.0069
Price_max: 0.0030
P